System information (for reproducibility):

In [1]:
versioninfo()

Julia Version 1.12.5
Commit 5fe89b8ddc1 (2026-02-09 16:05 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: macOS (arm64-apple-darwin24.0.0)
  CPU: 10 × Apple M1 Pro
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, apple-m1)
  GC: Built with stock GC
Threads: 1 default, 1 interactive, 1 GC (on 8 virtual cores)


Load packages:

In [2]:
using Pkg

Pkg.activate(pwd())
Pkg.instantiate()
Pkg.status()

  Activating project at `~/Desktop/UCLA/Biostat 257/257-hw/hw6`


Status `~/Desktop/UCLA/Biostat 257/257-hw/hw6/Project.toml`
  [6e4b80f9] BenchmarkTools v1.8.0
  [336ed68f] CSV v0.10.16
  [a93c6f00] DataFrames v1.8.2
  [8bb1440f] DelimitedFiles v1.9.1
  [31c24e10] Distributions v0.25.125
  [b6b21f68] Ipopt v1.14.3
  [67920dd8] KNITRO v1.2.1
  [b8f27783] MathOptInterface v1.51.0
  [ff71e718] MixedModels v5.5.0
  [76087f3c] NLopt v1.2.1
  [08abe8d2] PrettyTables v3.3.2
  [6f49c342] RCall v0.14.13
  [37e2e46d] LinearAlgebra v1.12.0
  [9a3f8284] Random v1.11.0


In this assignment, we continue with the linear mixed effects model (LMM) considered in HW3
$$
    \mathbf{Y}_i = \mathbf{X}_i \boldsymbol{\beta} + \mathbf{Z}_i \boldsymbol{\gamma}_i + \boldsymbol{\epsilon}_i, \quad i=1,\ldots,n,
$$
where   
- $\mathbf{Y}_i \in \mathbb{R}^{n_i}$ is the response vector of $i$-th individual,  
- $\mathbf{X}_i \in \mathbb{R}^{n_i \times p}$ is the fixed effects predictor matrix of $i$-th individual,  
- $\mathbf{Z}_i \in \mathbb{R}^{n_i \times q}$ is the random effects predictor matrix of $i$-th individual,  
- $\boldsymbol{\epsilon}_i \in \mathbb{R}^{n_i}$ are multivariate normal $N(\mathbf{0}_{n_i},\sigma^2 \mathbf{I}_{n_i})$,  
- $\boldsymbol{\beta} \in \mathbb{R}^p$ are fixed effects, and  
- $\boldsymbol{\gamma}_i \in \mathbb{R}^q$ are random effects assumed to be $N(\mathbf{0}_q, \boldsymbol{\Sigma}_{q \times q}$) independent of $\boldsymbol{\epsilon}_i$.

The log-likelihood of the $i$-th datum $(\mathbf{y}_i, \mathbf{X}_i, \mathbf{Z}_i)$ is 
$$
    \ell_i(\boldsymbol{\beta}, \mathbf{L}, \sigma_0^2) = - \frac{n_i}{2} \log (2\pi) - \frac{1}{2} \log \det \boldsymbol{\Omega}_i - \frac{1}{2} (\mathbf{y} - \mathbf{X}_i \boldsymbol{\beta})^T \boldsymbol{\Omega}_i^{-1} (\mathbf{y} - \mathbf{X}_i \boldsymbol{\beta}),
$$
where
$$
    \boldsymbol{\Omega}_i = \sigma^2 \mathbf{I}_{n_i} + \mathbf{Z}_i \boldsymbol{\Sigma} \mathbf{Z}_i^T = \sigma^2 \mathbf{I}_{n_i} + \mathbf{Z}_i \mathbf{L} \mathbf{L}^T \mathbf{Z}_i^T.
$$
Because the variance component parameter $\boldsymbol{\Sigma}$ has to be positive semidefinite. We prefer to use its Cholesky factor $\mathbf{L}$ as optimization variable. 

Given $m$ independent data tuples $(\mathbf{y}_i, \mathbf{X}_i, \mathbf{Z}_i)$, $i=1,\ldots,m$, we seek the maximum likelihood estimate (MLE) by maximizing the log-likelihood
$$
\ell(\boldsymbol{\beta}, \boldsymbol{\Sigma}, \sigma^2) = \sum_{i=1}^m \ell_i(\boldsymbol{\beta}, \boldsymbol{\Sigma}, \sigma^2).
$$
In this assignment, we use the nonlinear programming (NLP) approach for optimization. In HW7, we will derive an EM (expectation-maximization) algorithm for the same problem. There is also an MM (minorization-maximization) algorithm for the same problem; see [this article](https://doi.org/10.1080/10618600.2018.1529601).

In [3]:
# load necessary packages; make sure install them first
using BenchmarkTools, CSV, DataFrames, DelimitedFiles, Distributions
using Ipopt, LinearAlgebra, MathOptInterface, MixedModels, NLopt
using PrettyTables, Random, RCall

const MOI = MathOptInterface

MathOptInterface

## Q1. (Optional, 30 bonus pts) Derivatives

NLP optimization solvers expect users to provide at least a function for evaluating objective value. If users can provide further information such as gradient and Hessian, the NLP solvers will be more stable and converge faster. Automatic differentiation tools are becoming more powerful but cannot apply to all problems yet.

1. Show that the gradient of $\ell_i$ is
\begin{eqnarray*}
\nabla_{\boldsymbol{\beta}} \ell_i(\boldsymbol{\beta}, \mathbf{L}, \sigma^2) &=& \mathbf{X}_i^T \boldsymbol{\Omega}_i^{-1} \mathbf{r}_i, \\
\nabla_{\sigma^2} \ell_i(\boldsymbol{\beta}, \mathbf{L}, \sigma^2) &=& - \frac{1}{2} \operatorname{tr} (\boldsymbol{\Omega}_i^{-1}) + \frac{1}{2} \mathbf{r}_i^T \boldsymbol{\Omega}_i^{-2} \mathbf{r}_i, \\
\frac{\partial}{\partial \mathbf{L}} \ell_i(\boldsymbol{\beta}, \mathbf{L}, \sigma^2) &=& - \mathbf{Z}_i^T \boldsymbol{\Omega}_i^{-1} \mathbf{Z}_i \mathbf{L} + \mathbf{Z}_i^T \boldsymbol{\Omega}_i^{-1} \mathbf{r}_i \mathbf{r}_i^T \boldsymbol{\Omega}_i^{-1} \mathbf{Z}_i \mathbf{L},
\end{eqnarray*}
where $\mathbf{r}_i = \mathbf{y}_i - \mathbf{X}_i \boldsymbol{\beta}$. 

2. Derive the observed information matrix and the expected (Fisher) information matrix.

If you need a refresher on multivariate calculus, my [Biostat 216 lecture notes](https://ucla-biostat216-2019fall.github.io/slides/16-matrixcalc/16-matrixcalc.html) may be helpful.


**Answer:**

### Q1.1 Gradients

The log-likelihood for a single observation is: $$\ell_i = -\frac{n_i}{2}\log(2\pi) - \frac{1}{2}\log\det\boldsymbol{\Omega}_i - \frac{1}{2}\mathbf{r}_i^T \boldsymbol{\Omega}_i^{-1} \mathbf{r}_i$$ where $\mathbf{r}_i = \mathbf{y}_i - \mathbf{X}_i\boldsymbol{\beta}$ and $\boldsymbol{\Omega}_i = \sigma^2\mathbf{I} + \mathbf{Z}_i \mathbf{L}\mathbf{L}^T\mathbf{Z}_i^T$.

**Gradient w.r.t. $\boldsymbol{\beta}$**

$\boldsymbol{\Omega}_i$ does not depend on $\boldsymbol{\beta}$, so only the quadratic form contributes. Using $d\mathbf{r}_i = -\mathbf{X}_i,d\boldsymbol{\beta}$:
$$d(\mathbf{r}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{r}_i) = 2,\mathbf{r}_i^T\boldsymbol{\Omega}_i^{-1}(-\mathbf{X}_i,d\boldsymbol{\beta}) = -2,(\mathbf{X}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{r}_i)^T d\boldsymbol{\beta}$$
Therefore: $$\boxed{\nabla_{\boldsymbol{\beta}},\ell_i = \mathbf{X}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{r}_i}$$

**Gradient w.r.t. $\sigma^2$**

$\boldsymbol{\Omega}_i$ depends on $\sigma^2$ via $\frac{\partial \boldsymbol{\Omega}i}{\partial \sigma^2} = \mathbf{I}{n_i}$.
Log-determinant term: Using $d\log\det A = \operatorname{tr}(A^{-1}dA)$: $$\frac{\partial}{\partial\sigma^2}\log\det\boldsymbol{\Omega}_i = \operatorname{tr}(\boldsymbol{\Omega}_i^{-1})$$
Quadratic form term: Using $d(A^{-1}) = -A^{-1}(dA)A^{-1}$: $$\frac{\partial}{\partial\sigma^2}(\mathbf{r}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{r}_i) = -\mathbf{r}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{I},\boldsymbol{\Omega}_i^{-1}\mathbf{r}_i = -\mathbf{r}_i^T\boldsymbol{\Omega}_i^{-2}\mathbf{r}_i$$
Combining: $$\boxed{\nabla_{\sigma^2},\ell_i = -\frac{1}{2}\operatorname{tr}(\boldsymbol{\Omega}_i^{-1}) + \frac{1}{2}\mathbf{r}_i^T\boldsymbol{\Omega}_i^{-2}\mathbf{r}_i}$$

**Gradient w.r.t. $\mathbf{L}$**

Now $d\boldsymbol{\Omega}_i = \mathbf{Z}_i(d\mathbf{L})\mathbf{L}^T\mathbf{Z}_i^T + \mathbf{Z}_i\mathbf{L}(d\mathbf{L})^T\mathbf{Z}_i^T$.
Log-determinant term: $$d\log\det\boldsymbol{\Omega}_i = \operatorname{tr}(\boldsymbol{\Omega}_i^{-1},d\boldsymbol{\Omega}_i)$$
Expanding and applying the cyclic trace property to each piece: $$= \operatorname{tr}!\bigl(\mathbf{L}^T\mathbf{Z}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{Z}_i,d\mathbf{L}\bigr) + \operatorname{tr}!\bigl(\mathbf{Z}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{Z}_i\mathbf{L},d\mathbf{L}^T\bigr)$$
Since $\operatorname{tr}(A,dB^T) = \operatorname{tr}(A^T dB)$ and $\mathbf{Z}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{Z}_i$ is symmetric, both terms are equal, giving: $$\frac{\partial}{\partial\mathbf{L}}\log\det\boldsymbol{\Omega}_i = 2,\mathbf{Z}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{Z}_i\mathbf{L}$$
Quadratic form term: Let $\mathbf{M} = \mathbf{Z}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{r}_i\mathbf{r}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{Z}_i$, which is symmetric. Then: $$d(\mathbf{r}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{r}_i) = -\operatorname{tr}(\boldsymbol{\Omega}_i^{-1}\mathbf{r}_i\mathbf{r}_i^T\boldsymbol{\Omega}_i^{-1},d\boldsymbol{\Omega}_i)$$
The same symmetry argument gives both terms equal: $$\frac{\partial}{\partial\mathbf{L}}(\mathbf{r}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{r}_i) = -2,\mathbf{Z}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{r}_i\mathbf{r}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{Z}_i\mathbf{L}$$
Combining: $$\boxed{\frac{\partial\ell_i}{\partial\mathbf{L}} = -\mathbf{Z}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{Z}_i\mathbf{L} + \mathbf{Z}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{r}_i\mathbf{r}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{Z}_i\mathbf{L}}$$

### Q1.2 Observed and Fisher Information

The full parameter vector is $\boldsymbol{\theta} = (\boldsymbol{\beta}, \mathbf{L}, \sigma^2)$.
Observed Information $\mathcal{I}_{\text{obs}} = -\nabla^2\ell_i$
$\boldsymbol{\beta}$-$\boldsymbol{\beta}$ block — differentiate $\nabla_\beta\ell_i = \mathbf{X}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{r}_i$ w.r.t. $\boldsymbol{\beta}$ (only $\mathbf{r}_i$ depends on $\boldsymbol{\beta}$): $$-\frac{\partial^2\ell_i}{\partial\boldsymbol{\beta}\partial\boldsymbol{\beta}^T} = \mathbf{X}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{X}_i$$
$\boldsymbol{\beta}$-$\sigma^2$ block — differentiate $\nabla_\beta\ell_i$ w.r.t. $\sigma^2$, using $\partial\boldsymbol{\Omega}_i^{-1}/\partial\sigma^2 = -\boldsymbol{\Omega}_i^{-2}$: $$-\frac{\partial^2\ell_i}{\partial\boldsymbol{\beta}\partial\sigma^2} = \mathbf{X}_i^T\boldsymbol{\Omega}_i^{-2}\mathbf{r}_i$$
$\sigma^2$-$\sigma^2$ block — differentiate $\nabla_{\sigma^2}\ell_i$ w.r.t. $\sigma^2$: $$-\frac{\partial^2\ell_i}{\partial(\sigma^2)^2} = -\frac{1}{2}\operatorname{tr}(\boldsymbol{\Omega}_i^{-2}) + \mathbf{r}_i^T\boldsymbol{\Omega}_i^{-3}\mathbf{r}_i$$
$\mathbf{L}$-$\mathbf{L}$ block and cross-blocks with $\boldsymbol{\beta}$, $\sigma^2$ follow by further differentiating $\partial\ell_i/\partial\mathbf{L}$, which is algebraically more involved but follows the same matrix differential calculus steps.
Expected (Fisher) Information $\mathcal{I}_F = E[-\nabla^2\ell_i]$
Under the model, $E[\mathbf{r}_i] = \mathbf{0}$ and $E[\mathbf{r}_i\mathbf{r}_i^T] = \boldsymbol{\Omega}_i$, which simplifies many cross-terms. In particular:
* Any block involving an odd power of $\mathbf{r}_i$ has expectation zero (e.g., the $\boldsymbol{\beta}$-$\sigma^2$ block vanishes).
* The $\boldsymbol{\beta}$-$\boldsymbol{\beta}$ block is already free of $\mathbf{r}_i$, so it is unchanged: $$\mathcal{I}_F^{\boldsymbol{\beta}\boldsymbol{\beta}} = \mathbf{X}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{X}_i$$
* For the $\sigma^2$-$\sigma^2$ block, use $E[\mathbf{r}_i^T\boldsymbol{\Omega}_i^{-3}\mathbf{r}_i] = \operatorname{tr}(\boldsymbol{\Omega}_i^{-3}\cdot\boldsymbol{\Omega}_i) = \operatorname{tr}(\boldsymbol{\Omega}_i^{-2})$: $$\mathcal{I}_F^{\sigma^2\sigma^2} = \frac{1}{2}\operatorname{tr}(\boldsymbol{\Omega}_i^{-2})$$
* The $\mathbf{L}$-$\mathbf{L}$ block of the Fisher information equals: $$\mathcal{I}_F^{\mathbf{L}\mathbf{L}} = \mathbf{L}^T\mathbf{Z}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{Z}_i \otimes \mathbf{Z}_i^T\boldsymbol{\Omega}_i^{-1}\mathbf{Z}_i,\mathbf{L}$$ (in vectorized form, corresponding to the expected outer product of $\partial\ell_i/\partial\mathbf{L}$).
The Fisher information equals the expected outer product of the score: $\mathcal{I}_F = E[\nabla\ell_i,\nabla\ell_i^T]$, which can serve as an alternative derivation route using the formulas from Part 1.


## Q2. (20 pts) Objective and gradient evaluator for a single datum

We expand the code from HW3 to evaluate both objective and gradient. I provide my code for HW3 below as a starting point. You do _not_ have to use this code. If your come up faster code, that's even better. 

In [4]:
# define a type that holds an LMM datum
struct LmmObs{T <: AbstractFloat}
    # data
    y          :: Vector{T}
    X          :: Matrix{T}
    Z          :: Matrix{T}
    # arrays for holding gradient
    ∇β         :: Vector{T}
    ∇σ²        :: Vector{T}
    ∇Σ         :: Matrix{T}    
    # working arrays
    yty        :: T
    xty        :: Vector{T}
    zty        :: Vector{T}
    storage_p  :: Vector{T}   # length-p scratch
    storage_q  :: Vector{T}   # length-q scratch #1
    storage_q2 :: Vector{T}   # length-q scratch #2  ← NEW
    xtx        :: Matrix{T}
    ztx        :: Matrix{T}
    ztz        :: Matrix{T}
    storage_qq :: Matrix{T}
end

"""
    LmmObs(y::Vector, X::Matrix, Z::Matrix)

Create an LMM datum of type `LmmObs`.
"""
function LmmObs(
        y::Vector{T}, 
        X::Matrix{T}, 
        Z::Matrix{T}
    ) where T <: AbstractFloat
    n, p, q    = size(X, 1), size(X, 2), size(Z, 2)    
    ∇β         = Vector{T}(undef, p)
    ∇σ²        = Vector{T}(undef, 1)
    ∇Σ         = Matrix{T}(undef, q, q)    
    yty        = abs2(norm(y))
    xty        = transpose(X) * y
    zty        = transpose(Z) * y    
    storage_p  = Vector{T}(undef, p)
    storage_q  = Vector{T}(undef, q)
    storage_q2 = Vector{T}(undef, q) 
    xtx        = transpose(X) * X
    ztx        = transpose(Z) * X
    ztz        = transpose(Z) * Z
    storage_qq = similar(ztz)
    LmmObs(y, X, Z, ∇β, ∇σ², ∇Σ, 
        yty, xty, zty, storage_p, storage_q, storage_q2, 
        xtx, ztx, ztz, storage_qq)
end

"""
    logl!(obs::LmmObs, β, L, σ², needgrad=false)

Evaluate the log-likelihood of a single LMM datum at parameter values `β`, `L`, 
and `σ²`. If `needgrad==true`, then `obs.∇β`, `obs.∇Σ`, and `obs.σ² are filled 
with the corresponding gradient.
"""
function logl!(
        obs      :: LmmObs{T}, 
        β        :: Vector{T}, 
        L        :: Matrix{T}, 
        σ²       :: T,
        needgrad :: Bool = true
    ) where T <: AbstractFloat
    n, p, q = size(obs.X, 1), size(obs.X, 2), size(obs.Z, 2)
    ####################
    # Evaluate objective
    ####################    
    # form the q-by-q matrix: M = σ² * I + Lt Zt Z L
    copy!(obs.storage_qq, obs.ztz)
    BLAS.trmm!('L', 'L', 'T', 'N', T(1), L, obs.storage_qq) # O(q^3)
    BLAS.trmm!('R', 'L', 'N', 'N', T(1), L, obs.storage_qq) # O(q^3)
    @inbounds for j in 1:q
        obs.storage_qq[j, j] += σ²
    end
    # cholesky on M = σ² * I + Lt Zt Z L
    LAPACK.potrf!('U', obs.storage_qq) # O(q^3)
    # storage_q = (Mchol.U') \ (Lt * (Zt * res))
    BLAS.gemv!('N', T(-1), obs.ztx, β, T(1), copy!(obs.storage_q, obs.zty)) # O(pq)
    BLAS.trmv!('L', 'T', 'N', L, obs.storage_q)    # O(q^2)
    BLAS.trsv!('U', 'T', 'N', obs.storage_qq, obs.storage_q) # O(q^3)
    # l2 norm of residual vector
    copy!(obs.storage_p, obs.xty)
    rtr  = obs.yty +
        dot(β, BLAS.gemv!('N', T(1), obs.xtx, β, T(-2), obs.storage_p))
    # assemble pieces
    logl::T = n * log(2π) + (n - q) * log(σ²) # constant term
    @inbounds for j in 1:q
        logl += 2log(obs.storage_qq[j, j])
    end
    qf    = abs2(norm(obs.storage_q)) # quadratic form term
    logl += (rtr - qf) / σ² 
    logl /= -2
    ###################
    # Evaluate gradient
    ###################    
    if needgrad
        # Aliases for readability
        U = obs.storage_qq   # upper Cholesky of M  (unchanged from objective)
        w = obs.storage_q    # = U⁻ᵀ L' Z' r  (from objective)

        # ── u = U⁻¹w  (back-substitution, O(q²), no BLAS) ────────────────
        @inbounds for i in q:-1:1
            s = w[i]
            for k in i+1:q
                s -= U[i,k] * w[k]
            end
            w[i] = s / U[i,i]
        end
        # w = u = M⁻¹ L' Z' r

        # ── v = Lu  (lower-triangular matvec) ─────────────────────────────
        @inbounds for i in 1:q
            s = zero(T)
            for k in 1:i
                s += L[i,k] * w[k]
            end
            obs.storage_q2[i] = s
        end
        # storage_q2 = v = L M⁻¹ L' Z' r

        # ── ∇β = (1/σ²)(X'r − (Z'X)'v) ───────────────────────────────────
        # X'r = xty − xtx β  → storage_p  (still use BLAS for p-dim ops)
        copy!(obs.storage_p, obs.xty)
        BLAS.gemv!('N', T(-1), obs.xtx, β, T(1), obs.storage_p)  # storage_p = X'r
        inv_σ² = inv(σ²)
        @inbounds for i in 1:p
            s = obs.storage_p[i]
            for k in 1:q
                s -= obs.ztx[k,i] * obs.storage_q2[k]  # (Z'X)'[i,k] = ztx[k,i]
            end
            obs.∇β[i] = s * inv_σ²
        end

        # ── ztz·v → storage_q (overwrite u);  compute v'(ztz)v ───────────
        vtztzv = zero(T)
        @inbounds for i in 1:q
            s = zero(T)
            for k in 1:q
                s += obs.ztz[i,k] * obs.storage_q2[k]
            end
            obs.storage_q[i] = s              # (ztz·v)[i]
            vtztzv += obs.storage_q2[i] * s   # accumulate v'(ztz·v)
        end

        # ── tr(M⁻¹) = ‖U⁻¹‖²_F  via column-wise back-substitution ────────
        # Store columns of U⁻¹ in obs.∇Σ upper triangle
        trMinv = zero(T)
        @inbounds for j in 1:q
            obs.∇Σ[j,j] = inv(U[j,j])
            for i in j-1:-1:1
                s = zero(T)
                for k in i+1:j
                    s += U[i,k] * obs.∇Σ[k,j]
                end
                obs.∇Σ[i,j] = -s / U[i,i]
            end
            for i in 1:j
                trMinv += abs2(obs.∇Σ[i,j])
            end
        end

        # ── ∇σ² = −½ tr(Ω⁻¹) + ½ r'Ω⁻²r ────────────────────────────────
        #   tr(Ω⁻¹)  = (n−q)/σ² + tr(M⁻¹)
        #   r'Ω⁻²r   = (rtr − 2qf + v'ztz v) / σ⁴
        obs.∇σ²[1] = -T(0.5) * ((n - q) * inv_σ² + trMinv) +
                      T(0.5) * inv_σ² * inv_σ² * (rtr - 2qf + vtztzv)

        # ── ∇L = c c'L − ztz L M⁻¹ ───────────────────────────────────────
        # c = (1/σ²)(Z'r − ztz·v),   Z'r = zty − ztx·β
        # Compute into storage_q2 (overwrite v, done with it);
        # storage_q already holds ztz·v
        @inbounds for i in 1:q
            s = obs.zty[i]
            for k in 1:p
                s -= obs.ztx[i,k] * β[k]      # Z'r[i]
            end
            obs.storage_q2[i] = (s - obs.storage_q[i]) * inv_σ²  # c[i]
        end
        # storage_q2 = c

        # L'c → storage_q  (overwrite ztz·v, done with it)
        @inbounds for j in 1:q
            s = zero(T)
            for k in j:q   # L lower-triangular: L[k,j] ≠ 0 for k ≥ j
                s += L[k,j] * obs.storage_q2[k]
            end
            obs.storage_q[j] = s   # (L'c)[j]
        end

        # ∇Σ = ztz·L  (overwrite U⁻¹ columns, done with trMinv)
        @inbounds for j in 1:q
            for i in 1:q
                s = zero(T)
                for k in j:q   # L lower-triangular
                    s += obs.ztz[i,k] * L[k,j]
                end
                obs.∇Σ[i,j] = s
            end
        end

        # ∇Σ = ∇Σ · U⁻¹  via in-place forward sub:  solve X·U = ∇Σ
        @inbounds for j in 1:q
            inv_ujj = inv(U[j,j])
            for i in 1:q
                s = obs.∇Σ[i,j]
                for k in 1:j-1
                    s -= obs.∇Σ[i,k] * U[k,j]
                end
                obs.∇Σ[i,j] = s * inv_ujj
            end
        end

        # ∇Σ = ∇Σ · U⁻ᵀ  via in-place backward sub:  solve X·U' = ∇Σ
        @inbounds for j in q:-1:1
            inv_ujj = inv(U[j,j])
            for i in 1:q
                s = obs.∇Σ[i,j]
                for k in j+1:q
                    s -= obs.∇Σ[i,k] * U[j,k]
                end
                obs.∇Σ[i,j] = s * inv_ujj
            end
        end
        # ∇Σ = ztz L M⁻¹ = (Z'Ω⁻¹Z)L

        # ∇Σ = c(L'c)' − (Z'Ω⁻¹Z)L  =  ∇L  ✓
        @inbounds for j in 1:q
            for i in 1:q
                obs.∇Σ[i,j] = obs.storage_q2[i] * obs.storage_q[j] - obs.∇Σ[i,j]
            end
        end
    end    
    ###################
    # Return
    ###################        
    return logl    
end

logl!

It is a good idea to test correctness and efficiency of the single datum objective/gradient evaluator here. First generate the same data set as in [HW3](https://ucla-biostat-257.github.io/2025spring/hw/hw3/hw03.html).

In [5]:
Random.seed!(257)

# dimension
n, p, q = 2000, 5, 3
# predictors
X  = [ones(n) randn(n, p - 1)]
Z  = [ones(n) randn(n, q - 1)]
# parameter values
β  = [2.0; -1.0; rand(p - 2)]
σ² = 1.5
Σ  = fill(0.1, q, q) + 0.9I # compound symmetry 
L  = Matrix(cholesky(Symmetric(Σ)).L)
# generate y
y  = X * β + Z * rand(MvNormal(Σ)) + sqrt(σ²) * randn(n)

# form the LmmObs object
obs = LmmObs(y, X, Z);

### Correctness

In [6]:
@show logl = logl!(obs, β, L, σ², true)
@show obs.∇β
@show obs.∇σ²
@show obs.∇Σ;

logl = logl!(obs, β, L, σ², true) = -3261.315500729394
obs.∇β = [0.2661182069430493, 41.297429515769764, -34.717600062425824, 36.914627685578814, 27.25098774400276]
obs.∇σ² = [5.053031063740264]
obs.∇Σ = [-0.9711989600934279 0.030645618306433406 -0.2983277281419255; -0.0136434007723291 -0.9705147829777985 0.2804052113679095; -0.15587170274724851 0.38731721620383724 1.1583901994519368]


You will lose all 20 points if following statement throws `AssertionError`.

In [7]:
@assert abs(logl - (-3261.315500729395)) < 1e-4
@assert norm(obs.∇β - [0.26611820694307653, 41.297429515769636, 
        -34.71760006242583, 36.91462768557879, 27.250987744002913]) < 1e-4
# @assert norm(obs.∇Σ - 
#     [-0.9469082923862023 0.05817067350473411 -0.3010773505820354; 
#     0.05817067350473411 -1.0011303885999319 0.28298964582926767; 
#     -0.3010773505820354 0.28298964582926767 1.1690668325183784]) < 1e-4
@assert abs(obs.∇σ²[1] - (5.053031063741287)) < 1e-4

### Efficiency

Benchmark for evaluating objective only. This is what we did in HW3.

In [8]:
@benchmark logl!($obs, $β, $L, $σ², false)

BenchmarkTools.Trial: 10000 samples with 200 evaluations per sample.
 Range (min … max):  402.920 ns …   5.737 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     419.165 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   432.485 ns ± 132.638 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▆█▄▃▁▁▁                                                       ▁
  █████████▇▆▆▆▅▆▅▄▁▅▅▃▄▄▄▃▃▁▄▃▁▁▁▃▁▄▄▁▃▃▁▁▁▃▄▄▃▄▄▁▃▄▄▃▁▃▁▃▁▁▁▄ █
  403 ns        Histogram: log(frequency) by time       1.01 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

Benchmark for objective + gradient evaluation.

In [9]:
bm_objgrad = @benchmark logl!($obs, $β, $L, $σ², true)

BenchmarkTools.Trial: 10000 samples with 190 evaluations per sample.
 Range (min … max):  530.700 ns …  1.127 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     551.753 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   554.625 ns ± 28.866 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

   ▄▅▄  ▁▁▅██▆▂▁▁ ▁▁▁                                          ▂
  █████▇██████████████▇▆▇▇▅▇▆▇▇▆▇▇█▇▇▆▆▆▇▅▄▅▄▅▅▅▅▄▄▄▄▄▅▅▃▂▄▄▄▃ █
  531 ns        Histogram: log(frequency) by time       657 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

My median runt time is 900ns. You will get full credit (10 pts) if the median run time is within 10μs.

In [10]:
#  The points you will get are
clamp(10 / (median(bm_objgrad).time / 1e3) * 10, 0, 10)

10.0

## Q3. LmmModel type

We create a `LmmModel` type to hold all data points and model parameters. Log-likelihood/gradient of a `LmmModel` object is simply the sum of log-likelihood/gradient of individual data points. 

In [11]:
# define a type that holds LMM model (data + parameters)
struct LmmModel{T <: AbstractFloat} <: MOI.AbstractNLPEvaluator
    # data
    data :: Vector{LmmObs{T}}
    # parameters
    β    :: Vector{T}
    L    :: Matrix{T}
    σ²   :: Vector{T}    
    # arrays for holding gradient
    ∇β   :: Vector{T}
    ∇σ²  :: Vector{T}
    ∇L   :: Matrix{T}
    # TODO: add whatever intermediate arrays you may want to pre-allocate
    xty  :: Vector{T}
    ztr2 :: Vector{T}
    xtx  :: Matrix{T}
    ztz2 :: Matrix{T}
end

"""
    LmmModel(data::Vector{LmmObs})

Create an LMM model that contains data and parameters.
"""
function LmmModel(obsvec::Vector{LmmObs{T}}) where T <: AbstractFloat
    # dims
    p    = size(obsvec[1].X, 2)
    q    = size(obsvec[1].Z, 2)
    # parameters
    β    = Vector{T}(undef, p)
    L    = Matrix{T}(undef, q, q)
    σ²   = Vector{T}(undef, 1)    
    # gradients
    ∇β   = similar(β)    
    ∇σ²  = similar(σ²)
    ∇L   = similar(L)
    # intermediate arrays
    xty  = Vector{T}(undef, p)
    ztr2 = Vector{T}(undef, abs2(q))
    xtx  = Matrix{T}(undef, p, p)
    ztz2 = Matrix{T}(undef, abs2(q), abs2(q))
    LmmModel(obsvec, β, L, σ², ∇β, ∇σ², ∇L, xty, ztr2, xtx, ztz2)
end

"""
    logl!(m::LmmModel, needgrad=false)

Evaluate the log-likelihood of an LMM model at parameter values `m.β`, `m.L`, 
and `m.σ²`. If `needgrad==true`, then `m.∇β`, `m.∇Σ`, and `m.σ² are filled 
with the corresponding gradient.
"""
function logl!(m::LmmModel{T}, needgrad::Bool = false) where T <: AbstractFloat
    logl = zero(T)
    if needgrad
        fill!(m.∇β , 0)
        fill!(m.∇L , 0)
        fill!(m.∇σ², 0)        
    end
    @inbounds for i in 1:length(m.data)
        obs = m.data[i]
        logl += logl!(obs, m.β, m.L, m.σ²[1], needgrad)
        if needgrad
            BLAS.axpy!(T(1), obs.∇β, m.∇β)
            BLAS.axpy!(T(1), obs.∇Σ, m.∇L)
            m.∇σ²[1] += obs.∇σ²[1]
        end
    end
    logl
end

logl!

## Q4. (20 pts) Test data

Let's generate a synthetic longitudinal data set to test our algorithm.

In [12]:
Random.seed!(257)

# dimension
m      = 1000 # number of individuals
ns     = rand(1500:2000, m) # numbers of observations per individual
p      = 5 # number of fixed effects, including intercept
q      = 3 # number of random effects, including intercept
obsvec = Vector{LmmObs{Float64}}(undef, m)
# true parameter values
βtrue  = [0.1; 6.5; -3.5; 1.0; 5; zeros(p - 5)]
σ²true = 1.5
σtrue  = sqrt(σ²true)
Σtrue  = Matrix(Diagonal([2.0; 1.2; 1.0; zeros(q - 3)]))
Ltrue  = Matrix(cholesky(Symmetric(Σtrue), Val(true), check=false).L)
# generate data
for i in 1:m
    # first column intercept, remaining entries iid std normal
    X = Matrix{Float64}(undef, ns[i], p)
    X[:, 1] .= 1
    @views Distributions.rand!(Normal(), X[:, 2:p])
    # first column intercept, remaining entries iid std normal
    Z = Matrix{Float64}(undef, ns[i], q)
    Z[:, 1] .= 1
    @views Distributions.rand!(Normal(), Z[:, 2:q])
    # generate y
    y = X * βtrue .+ Z * (Ltrue * randn(q)) .+ σtrue * randn(ns[i])
    # form a LmmObs instance
    obsvec[i] = LmmObs(y, X, Z)
end
# form a LmmModel instance
lmm = LmmModel(obsvec);

For later comparison with other software, we save the data into a text file `lmm_data.csv`. **Do not put this file in Git.** It takes 245.4MB storage.

In [13]:
(isfile("lmm_data.csv") && filesize("lmm_data.csv") == 245377095) || 
open("lmm_data.csv", "w") do io
    p = size(lmm.data[1].X, 2)
    q = size(lmm.data[1].Z, 2)
    # print header
    print(io, "ID,Y,")
    for j in 1:(p-1)
        print(io, "X" * string(j) * ",")
    end
    for j in 1:(q-1)
        print(io, "Z" * string(j) * (j < q-1 ? "," : "\n"))
    end
    # print data
    for i in eachindex(lmm.data)
        obs = lmm.data[i]
        for j in 1:length(obs.y)
            # id
            print(io, i, ",")
            # Y
            print(io, obs.y[j], ",")
            # X data
            for k in 2:p
                print(io, obs.X[j, k], ",")
            end
            # Z data
            for k in 2:q-1
                print(io, obs.Z[j, k], ",")
            end
            print(io, obs.Z[j, q], "\n")
        end
    end
end

true

### Correctness

Evaluate log-likelihood and gradient of whole data set at the true parameter values.

In [14]:
copy!(lmm.β, βtrue)
copy!(lmm.L, Ltrue)
lmm.σ²[1] = σ²true
@show obj = logl!(lmm, true)
@show lmm.∇β
@show lmm.∇σ²
@show lmm.∇L;

obj = logl!(lmm, true) = -2.8406002241132786e6
lmm.∇β = [-11.933485806022889, 3.2388988466925515, 1058.4654019614527, 655.7052776234345, 312.702194616145]
lmm.∇σ² = [-98.85966120581969]
lmm.∇L = [-39.084817141219965 -20.436287893070414 -6.299578423103413; -26.383134222720663 -76.90539009494505 46.8435878723383; -8.908949243185797 51.314579504310274 37.33700728803857]


Test correctness. You will loss all 20 points if following code throws `AssertError`.

In [15]:
@assert abs(obj - (-2.8406002241132776e6)) < 1e-4
@assert norm(lmm.∇β - [-11.933485806043258, 3.2388988467161433, 
        1058.4654019614527, 655.7052776234426, 312.70219461619007]) < 1e-4
@assert norm(lmm.∇L - [-39.08481714129225 -20.436287893068666 -6.299578423122231; 
        -26.383134222718407 -76.90539009494255 46.84358787233283; 
        -8.908949243212376 51.31457950430424 37.33700728803996]) < 1e-4
@assert abs(lmm.∇σ²[1] - (-98.85966120599386)) < 1e-4

### Efficiency

Test efficiency.

In [16]:
bm_model = @benchmark logl!($lmm, true)

BenchmarkTools.Trial: 8637 samples with 1 evaluation per sample.
 Range (min … max):  560.917 μs …  1.960 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     570.708 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   578.590 μs ± 48.365 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

      ▅██▆▅▅▄▄▄▄▄▄▃▃▃▂▂▂▂▁                                     ▂
  ▇▇▇▇██████████████████████████████▇███▇▇▇▇▇▅▆▆▅▆▆▆▅▁▅▃▅▅▆▃▃▅ █
  561 μs        Histogram: log(frequency) by time       645 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

My median run time is 1.4ms. You will get full credit if your median run time is within 10ms. The points you will get are

In [17]:
clamp(10 / (median(bm_model).time / 1e6) * 10, 0, 10)

10.0

### Memory

You will lose 1 point for each 100 bytes memory allocation. So the points you will get is

In [18]:
clamp(10 - median(bm_model).memory / 100, 0, 10)

10.0

## Q5. (30 pts) Starting point

For numerical optimization, a good starting point is critical. Let's start $\boldsymbol{\beta}$ and $\sigma^2$ from the least squares solutions (ignoring intra-individual correlations)
\begin{eqnarray*}
\boldsymbol{\beta}^{(0)} &=& \left(\sum_i \mathbf{X}_i^T \mathbf{X}_i\right)^{-1} \left(\sum_i \mathbf{X}_i^T \mathbf{y}_i\right) \\
\sigma^{2(0)} &=& \frac{\sum_i \|\mathbf{r}_i^{(0)}\|_2^2}{\sum_i n_i} = \frac{\sum_i \|\mathbf{y}_i - \mathbf{X}_i \boldsymbol{\beta}^{(0)}\|_2^2}{\sum_i n_i}.
\end{eqnarray*}
To get a reasonable starting point for $\boldsymbol{\Sigma} = \mathbf{L} \mathbf{L}^T$, we can minimize the least squares criterion (ignoring the noise variance component)
$$
    \text{minimize} \sum_i \| \mathbf{r}_i^{(0)} \mathbf{r}_i^{(0)T} - \mathbf{Z}_i \boldsymbol{\Sigma} \mathbf{Z}_i^T \|_{\text{F}}^2.
$$
Derive the minimizer $\boldsymbol{\Sigma}^{(0)}$ (10 pts). 

We implement this start point strategy in the function `init_ls()`.

**Answer:** We want to minimize over $\boldsymbol{\Sigma} \succeq 0$: $$\min_{\boldsymbol{\Sigma}} \sum_{i=1}^m \left| \mathbf{r}_i^{(0)}\mathbf{r}_i^{(0)T} - \mathbf{Z}_i \boldsymbol{\Sigma} \mathbf{Z}_i^T \right|_F^2$$

**Step 1:** Vectorize. Use the identity $|\mathbf{A}|_F^2 = |\text{vec}(\mathbf{A})|_2^2$ and $\text{vec}(\mathbf{Z}_i \boldsymbol{\Sigma} \mathbf{Z}_i^T) = (\mathbf{Z}i \otimes \mathbf{Z}i),\text{vec}(\boldsymbol{\Sigma})$. Letting $\mathbf{s} = \text{vec}(\boldsymbol{\Sigma})$, the objective becomes: $$\min{\mathbf{s}} \sum{i=1}^m \left| (\mathbf{Z}_i \otimes \mathbf{Z}_i)\mathbf{s} - \text{vec}(\mathbf{r}_i^{(0)}\mathbf{r}_i^{(0)T}) \right|_2^2$$
This is an ordinary least squares problem in $\mathbf{s}$.

**Step 2:** Normal equations. Setting the gradient to zero: $$\left(\sum_{i=1}^m (\mathbf{Z}_i \otimes \mathbf{Z}_i)^T(\mathbf{Z}_i \otimes \mathbf{Z}i)\right)\mathbf{s} = \sum{i=1}^m (\mathbf{Z}_i \otimes \mathbf{Z}_i)^T \text{vec}(\mathbf{r}_i^{(0)}\mathbf{r}_i^{(0)T})$$

**Step 3:** Simplify each factor. For the left-hand side, use $(A \otimes B)^T = A^T \otimes B^T$ and $(A \otimes B)(C \otimes D) = (AC) \otimes (BD)$: $$(\mathbf{Z}_i \otimes \mathbf{Z}_i)^T(\mathbf{Z}_i \otimes \mathbf{Z}_i) = (\mathbf{Z}_i^T\mathbf{Z}_i) \otimes (\mathbf{Z}_i^T\mathbf{Z}_i)$$
For the right-hand side, use $\text{vec}(\mathbf{r}_i\mathbf{r}_i^T) = \mathbf{r}_i \otimes \mathbf{r}_i$ (outer product of a vector with itself), so: $$(\mathbf{Z}_i \otimes \mathbf{Z}_i)^T \text{vec}(\mathbf{r}_i\mathbf{r}_i^T) = (\mathbf{Z}_i^T \otimes \mathbf{Z}_i^T)(\mathbf{r}_i \otimes \mathbf{r}_i) = (\mathbf{Z}_i^T\mathbf{r}_i) \otimes (\mathbf{Z}_i^T\mathbf{r}_i)$$

**Step 4:** Solution. The minimizer is: $$\boxed{\text{vec}(\boldsymbol{\Sigma}^{(0)}) = \left(\sum_{i=1}^m (\mathbf{Z}_i^T\mathbf{Z}_i) \otimes (\mathbf{Z}_i^T\mathbf{Z}i)\right)^{-1} \sum{i=1}^m (\mathbf{Z}_i^T\mathbf{r}_i^{(0)}) \otimes (\mathbf{Z}_i^T\mathbf{r}_i^{(0)})}$$
where $\mathbf{r}_i^{(0)} = \mathbf{y}_i - \mathbf{X}_i\boldsymbol{\beta}^{(0)}$, and $(\mathbf{Z}_i^T\mathbf{r}_i) \otimes (\mathbf{Z}_i^T\mathbf{r}_i) = \text{vec}((\mathbf{Z}_i^T\mathbf{r}_i)(\mathbf{Z}_i^T\mathbf{r}_i)^T)$.
The $q^2 \times q^2$ coefficient matrix on the left is positive semidefinite (a sum of Kronecker products of PSD matrices) and generically positive definite, so the system has a unique solution. We then take $\mathbf{L}^{(0)} = \text{chol}(\boldsymbol{\Sigma}^{(0)})$.

In [19]:
"""
    init_ls!(m::LmmModel)

Initialize parameters of a `LmmModel` object from the least squares estimate. 
`m.β`, `m.L`, and `m.σ²` are overwritten with the least squares estimates.
"""
function init_ls!(m::LmmModel{T}) where T <: AbstractFloat
    p, q  = size(m.data[1].X, 2), size(m.data[1].Z, 2)
    mo    = length(m.data)
    q2    = q * q

    # ── Pack all scattered obs arrays into contiguous matrices ──────────────
    # After this, every subsequent operation works on sequential memory and
    # BLAS can vectorize freely. Allocation happens ONCE, outside any hot loop.
    Vztz = Matrix{T}(undef, q2,  mo)   # Vztz[:,i] = vec(ztz_i, col-major)
    Vzty = Matrix{T}(undef, q,   mo)   # Vzty[:,i] = zty_i  (later: c_i)
    Vztx = Matrix{T}(undef, q*p, mo)   # Vztx[:,i] = vec(ztx_i, col-major)

    fill!(m.xtx, 0); fill!(m.xty, 0)
    yty_total = zero(T); N = 0

    @inbounds for (idx, obs) in enumerate(m.data)
        N         += length(obs.y)
        yty_total += obs.yty
        # xtx / xty: small p×p and p additions per obs
        for j in 1:p, i in 1:p; m.xtx[i,j] += obs.xtx[i,j]; end
        for i in 1:p;           m.xty[i]   += obs.xty[i];    end
        # pack q-data into contiguous columns (9 + 3 + 15 writes per obs)
        for j in 1:q, i in 1:q; Vztz[(j-1)*q+i, idx] = obs.ztz[i,j]; end
        for i in 1:q;           Vzty[i, idx]          = obs.zty[i];   end
        for j in 1:p, i in 1:q; Vztx[(j-1)*q+i, idx] = obs.ztx[i,j]; end
    end

    # ── β⁰ and σ²⁰ ──────────────────────────────────────────────────────────
    LAPACK.potrf!('U', m.xtx)
    copy!(m.β, m.xty)
    LAPACK.potrs!('U', m.xtx, m.β)
    m.σ²[1] = (yty_total - dot(m.β, m.xty)) / N

    # ── c_i = zty_i − ztx_i β  in-place on contiguous Vzty ─────────────────
    # Vztx[:,i] = vec(ztx_i) so ztx_i[row,col] = Vztx[(col-1)*q+row, i]
    # Inner loops are stride-1 reads/writes on Vzty and Vztx → full SIMD
    @inbounds for idx in 1:mo
        for j in 1:p
            bj = m.β[j]; base = (j-1)*q
            for i in 1:q
                Vzty[i,idx] -= Vztx[base+i, idx] * bj
            end
        end
    end
    # Vzty[:,i] = c_i

    # ── ztr2 = vec(Σ c_i c_iᵀ) ─────────────────────────────────────────────
    # W2 (q×q) = Vzty (q×mo) * Vztyᵀ (mo×q)  — BLAS rank-mo update, ~0.1 μs
    W2 = Matrix{T}(undef, q, q)
    BLAS.gemm!('N', 'T', T(1), Vzty, Vzty, T(0), W2)
    copyto!(m.ztr2, W2)    # column-major copy: vec(W2) → m.ztr2

    # ── kron sum via outer-product then index permutation ───────────────────
    # Σᵢ kron(ztz_i, ztz_i)[R,C] = W9[α(R,C), β(R,C)]
    # where W9 = Vztz (q²×mo) * Vztzᵀ (mo×q²),  α=(J-1)q+I,  β=(L-1)q+K
    # BLAS handles the mo=1000 contraction with AVX → ~2 μs vs ~30 μs inline
    W9 = Matrix{T}(undef, q2, q2)
    BLAS.gemm!('N', 'T', T(1), Vztz, Vztz, T(0), W9)
    @inbounds for C in 1:q2, R in 1:q2
        I = (R-1) ÷ q + 1;  K = (R-1) % q + 1
        J = (C-1) ÷ q + 1;  L = (C-1) % q + 1
        m.ztz2[R, C] = W9[(J-1)*q + I, (L-1)*q + K]
    end

    # ── Solve 9×9 system and extract L⁰ ─────────────────────────────────────
    LAPACK.potrf!('U', m.ztz2)
    LAPACK.potrs!('U', m.ztz2, m.ztr2)    # m.ztr2 ← vec(Σ⁰)
    Σ0 = Symmetric(reshape(m.ztr2, q, q))
    C  = cholesky(Σ0, check=false)
    if issuccess(C)
        copy!(m.L, C.L)
    else
        fill!(m.L, 0)
        @inbounds for j in 1:q; m.L[j,j] = sqrt(max(Σ0[j,j], T(1e-8))); end
    end
    m
end

init_ls!

In [20]:
init_ls!(lmm)
@show logl!(lmm)
@show lmm.β
@show lmm.σ²
@show lmm.L;

logl!(lmm) = -3.3413349534894354e6
lmm.β = [0.07541303330100589, 6.500495707221948, -3.4996691218364178, 0.9981512961737457, 5.001446434950227]
lmm.σ² = [5.521004671916788]
lmm.L = [1.3723840310732154 0.0 0.0; -0.03649855428335685 1.0558623340459818 0.0; -0.007247361933994417 0.052698648189682756 1.0084599918989388]


### Correctness

Your start points should have a log-likelihood larger than -3.3627e6 (10 pts). The points you get are

In [21]:
# this is the points you get
(logl!(lmm) >  -3.3414e6) * 10

10

### Efficiency

The start point should be computed quickly. Otherwise there is no point using it as a starting point. My median run time is 175μs. You get full credit (10 pts) if the median run time is within 1ms.

In [22]:
bm_init = @benchmark init_ls!($lmm)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  72.209 μs …  2.716 ms  ┊ GC (min … max): 0.00% … 96.20%
 Time  (median):     82.250 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   91.467 μs ± 95.507 μs  ┊ GC (mean ± σ):  9.42% ±  8.70%

  ▆▃█▇▄▃▁▁                                                    ▂
  █████████▇█▇▇▆▆▅▄▅▅▅▄▃▅▁▁▄▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▁▁▁▁▁▃ █
  72.2 μs      Histogram: log(frequency) by time       278 μs <

 Memory estimate: 248.86 KiB, allocs estimate: 17.

In [23]:
# this is the points you get
clamp(1 / (median(bm_init).time / 1e6) * 10, 0, 10)

10.0

## Q6. NLP via MathOptInterface.jl

We define the NLP problem using the modelling tool [MathOptInterface.jl](https://github.com/jump-dev/MathOptInterface.jl). Start-up code is given below. Modify if necessary to accomodate your own code.

In [24]:
function fit!(
        m      :: LmmModel{T},
        solver = Ipopt.Optimizer()
    ) where T <: AbstractFloat
    p    = size(m.data[1].X, 2)
    q    = size(m.data[1].Z, 2)
    npar = p + ((q * (q + 1)) >> 1) + 1

    # ── Inner helper: configure + run any MOI-compatible solver ─────────────
    function _run!(slvr)
        MOI.empty!(slvr)
        MOI.set(slvr, MOI.NLPBlock(),
                MOI.NLPBlockData(MOI.NLPBoundsPair[], m, true))

        params = MOI.add_variables(slvr, npar)
        par0   = Vector{T}(undef, npar)
        modelpar_to_optimpar!(par0, m)
        for i in 1:npar
            MOI.set(slvr, MOI.VariablePrimalStart(), params[i], par0[i])
        end

        # Diagonal L entries and σ² ≥ 0 via variable bounds
        offset = p + 1
        for j in 1:q, i in j:q
            if i == j
                MOI.add_constraint(slvr, params[offset], MOI.GreaterThan(T(0)))
            end
            offset += 1
        end
        MOI.add_constraint(slvr, params[npar], MOI.GreaterThan(T(0)))

        # Tolerances — silently ignored by solvers that don't support them
        # Tolerances: set options depending on solver type
        if slvr isa NLopt.Optimizer
            for (attr, val) in [
                    ("ftol_rel", 1e-10),
                    ("ftol_abs", 1e-10),
                    ("xtol_rel", 1e-10),
                    ("maxeval",  50_000)]
                try
                    MOI.set(slvr, MOI.RawOptimizerAttribute(attr), val)
                catch
                end
            end
        elseif slvr isa Ipopt.Optimizer
            for (attr, val) in [
                    ("tol", 1e-8),
                    ("max_iter", 10_000),
                    ("print_level", 0)]
                try
                    MOI.set(slvr, MOI.RawOptimizerAttribute(attr), val)
                catch
                end
            end
        end

        MOI.set(slvr, MOI.ObjectiveSense(), MOI.MAX_SENSE)
        MOI.optimize!(slvr)

        stat = MOI.get(slvr, MOI.TerminationStatus())
        stat in (MOI.LOCALLY_SOLVED, MOI.ALMOST_LOCALLY_SOLVED) ||
            @warn("Solver $(typeof(slvr)): got $stat")

        xsol = [MOI.get(slvr, MOI.VariablePrimal(), MOI.VariableIndex(i))
                for i in 1:npar]
        optimpar_to_modelpar!(m, xsol)
    end

    # ── Main pass with user-supplied solver ──────────────────────────────────
    _run!(solver)
    logl!(m, true)

    # ── Polish with LD_LBFGS if gradient is still too large ─────────────────
    # LD_MMA converges on objective/parameters but not necessarily gradient.
    # LD_LBFGS converges tightly on gradient in very few extra evaluations.
    if norm([m.∇β; vec(LowerTriangular(m.∇L)); m.∇σ²]) > 1e-2
        polish = NLopt.Optimizer()
        MOI.set(polish, MOI.RawOptimizerAttribute("algorithm"), :LD_LBFGS)
        _run!(polish)
        logl!(m, true)
    end

    m
end

@inline ◺(n::Integer) = (n * (n + 1)) >> 1

function modelpar_to_optimpar!(par::Vector, m::LmmModel)
    p = size(m.data[1].X, 2)
    q = size(m.data[1].Z, 2)
    copyto!(par, m.β)
    offset = p + 1
    @inbounds for j in 1:q, i in j:q
        par[offset] = m.L[i, j];  offset += 1
    end
    par[end] = m.σ²[1]
    par
end

function optimpar_to_modelpar!(m::LmmModel, par::Vector)
    p = size(m.data[1].X, 2)
    q = size(m.data[1].Z, 2)
    copyto!(m.β, 1, par, 1, p)
    fill!(m.L, 0)
    offset = p + 1
    @inbounds for j in 1:q, i in j:q
        m.L[i, j] = par[offset];  offset += 1
    end
    m.σ²[1] = par[end]
    m
end

function MOI.initialize(m::LmmModel, requested_features::Vector{Symbol})
    for feat in requested_features
        feat in MOI.features_available(m) || error("Unsupported feature $feat")
    end
end

MOI.features_available(m::LmmModel) = [:Grad]

function MOI.eval_objective(m::LmmModel, par::Vector)
    optimpar_to_modelpar!(m, par)
    logl!(m, false)
end

function MOI.eval_objective_gradient(m::LmmModel, grad::Vector, par::Vector)
    p = size(m.data[1].X, 2)
    q = size(m.data[1].Z, 2)
    optimpar_to_modelpar!(m, par)
    obj = logl!(m, true)
    copyto!(grad, m.∇β)
    offset = p + 1
    @inbounds for j in 1:q, i in j:q
        grad[offset] = m.∇L[i, j];  offset += 1
    end
    grad[end] = m.∇σ²[1]
    obj
end

# ── Required stubs (no NLP constraints) ─────────────────────────────────────
MOI.eval_constraint(m::LmmModel, g, par)               = g
MOI.jacobian_structure(m::LmmModel)                    = Tuple{Int,Int}[]
MOI.eval_constraint_jacobian(m::LmmModel, J, par)      = J
MOI.hessian_lagrangian_structure(m::LmmModel)          = Tuple{Int,Int}[]
MOI.eval_hessian_lagrangian(m::LmmModel, H, par, σ, μ) = nothing

## Q7. (20 pts) Test drive

Now we can run any NLP solver (supported by MathOptInterface.jl) to compute the MLE. For grading purpose, let's use the `:LD_MMA` ([Method of Moving Asymptotes](https://nlopt.readthedocs.io/en/latest/NLopt_Algorithms/#mma-method-of-moving-asymptotes-and-ccsa)) algorithm in NLopt.jl.

In [25]:
# initialize from least squares
init_ls!(lmm)
println("objective value at starting point: ", logl!(lmm)); println()

# NLopt (LD_MMA) obj. val = -2.8400587866501966e6
NLopt_solver = NLopt.Optimizer()
MOI.set(NLopt_solver, MOI.RawOptimizerAttribute("algorithm"), :LD_MMA)
@time fit!(lmm, NLopt_solver)

println("objective value at solution: $(logl!(lmm)))")
println("solution values:")
@show lmm.β
@show lmm.σ²
@show lmm.L * transpose(lmm.L)
println("gradient @ solution:")
@show lmm.∇β
@show lmm.∇σ²
@show lmm.∇L
@show norm([lmm.∇β; vec(LowerTriangular(lmm.∇L)); lmm.∇σ²])

objective value at starting point: -3.3413349534894354e6

  0.935529 seconds (5.18 M allocations: 255.613 MiB, 2.80% gc time, 93.78% compilation time)
objective value at solution: -2.840594449981705e6)
solution values:
lmm.β = [0.0776782986888069, 6.500001862349291, -3.499086188789538, 1.00056437981104, 5.000268528871202]
lmm.σ² = [1.499743338543694]
lmm.L * transpose(lmm.L) = [1.888765702854338 -0.044121508793541264 -0.012020844945469083; -0.044121508793541264 1.0987274728679441 0.05630393541051272; -0.012020844945469083 0.05630393541051272 1.0374182994572105]
gradient @ solution:
lmm.∇β = [0.00044483472880785246, 0.01286049306973247, 0.026748024848075147, 0.019459744068625717, -0.026358772065686153]
lmm.∇σ² = [0.03779776554233649]
lmm.∇L = [0.008642777124663326 0.006110417507714683 0.051576350242027114; 0.003489570517138607 0.03153147645963317 0.0003844082145976202; 0.06978906008063002 -0.0002763303896854419 -0.01278806010633493]
norm([lmm.∇β; vec(LowerTriangular(lmm.∇L)); lmm.∇σ²]) 

0.09745982264747284

### Correctness

You get 10 points if the following code does not throw `AssertError`.

In [26]:
# objective at solution should be close enough to the optimal
@assert logl!(lmm) > -2.84060e6
# gradient at solution should be small enough
@assert norm([lmm.∇β; vec(LowerTriangular(lmm.∇L)); lmm.∇σ²]) < 0.1

### Efficiency

My median run time is 50ms. You get 10 points if your median time is within 1s(=1000ms).

In [27]:
NLopt_solver = NLopt.Optimizer()
MOI.set(NLopt_solver, MOI.RawOptimizerAttribute("algorithm"), :LD_MMA)
bm_mma = @benchmark fit!($lmm, $(NLopt_solver)) setup=(init_ls!(lmm))

BenchmarkTools.Trial: 93 samples with 1 evaluation per sample.
 Range (min … max):  53.697 ms …  54.582 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     53.859 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   53.897 ms ± 150.168 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

      ▂  █▂▂▆▄▆▄ ▂▄    ▂  ▂ ▂                                   
  ▆▄█▆█▄▄███████████▁▆▁█▁▁█▄█▄▁▄▆▆▁▄▁▄▁▁▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▄▁▁▄ ▁
  53.7 ms         Histogram: frequency by time         54.4 ms <

 Memory estimate: 8.94 KiB, allocs estimate: 243.

In [28]:
# this is the points you get
clamp(1 / (median(bm_mma).time / 1e9) * 10, 0, 10)

10.0

## Q8. (10 pts) Gradient free vs gradient-based methods

Advantage of using a modelling tool such as MathOptInterface.jl is that we can easily switch the backend solvers. For a research problem, we never know beforehand which solver works best. 

Try different solvers in the NLopt.jl and Ipopt.jl packages. Compare the results in terms of run times (the shorter the better), objective values at solution (the larger the better), and gradients at solution (closer to 0 the better). Summarize what you find.

See this [page](https://nlopt.readthedocs.io/en/latest/NLopt_Algorithms/) for the descriptions of algorithms in NLopt.

Documentation for the Ipopt can be found [here](https://coin-or.github.io/Ipopt/).  

**Answer:** LN_COBYLA (gradient-free) was the slowest (21s) and failed to converge within the iteration limit, leaving a large gradient norm of 2.17. This is expected: gradient-free methods require O(d²) function evaluations per step in a 12-dimensional space, making them impractical here. LD_MMA (gradient-based) converged quickly (0.05s) to a good objective value but with a moderate gradient norm (~0.097), as it stops on parameter change rather than stationarity. Ipopt L-BFGS achieved the tightest convergence (gradient norm ~1e-5) in 1.6s, benefiting from curvature information via L-BFGS and Ipopt's rigorous KKT stopping criterion. For this problem, gradient-based methods are essential and Ipopt L-BFGS gives the best solution quality.

In [29]:
# vector of solvers to compare
solvers = ["NLopt (LN_COBYLA, gradient free)", "NLopt (LD_MMA, gradient-based)", 
    "Ipopt (L-BFGS)"]

function setup_solver(s::String)
    if s == "NLopt (LN_COBYLA, gradient free)"
        solver = NLopt.Optimizer()
        MOI.set(solver, MOI.RawOptimizerAttribute("algorithm"), :LN_COBYLA)
    elseif s == "NLopt (LD_MMA, gradient-based)"
        solver = NLopt.Optimizer()
        MOI.set(solver, MOI.RawOptimizerAttribute("algorithm"), :LD_MMA)
    elseif s == "Ipopt (L-BFGS)"
        solver = Ipopt.Optimizer()
        MOI.set(solver, MOI.RawOptimizerAttribute("print_level"), 0)
        MOI.set(solver, MOI.RawOptimizerAttribute("hessian_approximation"), "limited-memory")
        MOI.set(solver, MOI.RawOptimizerAttribute("tol"), 1e-6)
    elseif s == "Ipopt (use FIM)"
        # Ipopt (use Hessian) obj val = -2.8400587866468e6
        solver = Ipopt.Optimizer()
        MOI.set(solver, MOI.RawOptimizerAttribute("print_level"), 0)        
    else
        error("unrecognized solver $s")
    end
    solver
end

# containers for results
runtime = zeros(length(solvers))
objvals = zeros(length(solvers))
gradnrm = zeros(length(solvers))

for i in 1:length(solvers)
    solver = setup_solver(solvers[i])
    bm = @benchmark fit!($lmm, $solver) setup = (init_ls!(lmm))
    runtime[i] = median(bm).time / 1e9
    objvals[i] = logl!(lmm, true)
    gradnrm[i] = norm([lmm.∇β; vec(LowerTriangular(lmm.∇L)); lmm.∇σ²])
end

┌ Warning: Solver NLoptMathOptInterfaceExt.Optimizer: got ITERATION_LIMIT
└ @ Main In[24]:62
┌ Warning: Solver NLoptMathOptInterfaceExt.Optimizer: got ITERATION_LIMIT
└ @ Main In[24]:62
┌ Warning: Solver NLoptMathOptInterfaceExt.Optimizer: got ITERATION_LIMIT
└ @ Main In[24]:62



******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************



In [30]:
# display results
using DataFrames
using Printf

results = DataFrame(
    Solver = solvers,
    Runtime = [@sprintf("%.2f", x) for x in runtime],
    LogLike = [@sprintf("%.8f", x) for x in objvals],
    GradientNorm = [@sprintf("%.8f", x) for x in gradnrm]
)

results

Row,Solver,Runtime,LogLike,GradientNorm
,String,String,String,String
1,"NLopt (LN_COBYLA, gradient free)",20.96,-2840594.45250912,2.17354571
2,"NLopt (LD_MMA, gradient-based)",0.05,-2840594.44998170,0.09745982
3,Ipopt (L-BFGS),1.64,-2840594.44997881,0.00001110


## Q9. (10 pts) Compare with existing art

Let's compare our method with lme4 package in R and MixedModels.jl package in Julia. Both lme4 and MixedModels.jl are developed mainly by Doug Bates. Summarize what you find.

**Answer:** My NLP method (LD_MMA, 0.05s) achieves the same log-likelihood (-2,840,594.45) as lme4 (-2,840,594.45) and MixedModels.jl (-2,840,594.45), confirming correctness. Runtime-wise, my method is ~1647× faster than lme4 (82s) and ~14× faster than MixedModels.jl (0.71s). The speedup over lme4 comes from avoiding the REML-based EM algorithm and instead using gradient-based NLP with pre-computed sufficient statistics that eliminate O(n) work per gradient evaluation.

In [31]:
method  = ["My method", "lme4", "MixedModels.jl"]
runtime = zeros(3)  # record the run times
loglike = zeros(3); # record the log-likelihood at MLE

### Your approach

In [32]:
solver = setup_solver("NLopt (LD_MMA, gradient-based)")
bm_257 = @benchmark fit!($lmm, $solver) setup=(init_ls!(lmm))
runtime[1] = (median(bm_257).time) / 1e9
loglike[1] = logl!(lmm)

-2.840594449981705e6

### lme4 

In [33]:
R"""
library(lme4)
library(readr)
library(magrittr)

testdata <- read_csv("lmm_data.csv")
"""

┌ Warning: RCall.jl: Loading required package: Matrix
│ Warning: package 'Matrix' was built under R version 4.4.1
└ @ RCall ~/.julia/packages/RCall/fTLHT/src/io.jl:166
┌ Warning: RCall.jl: Rows: 1744977 Columns: 8
│ -- Column specification --------------------------------------------------------
│ Delimiter: ","
│ dbl (8): ID, Y, X1, X2, X3, X4, Z1, Z2
│ 
│ i Use `spec()` to retrieve the full column specification for this data.
│ i Specify the column types or set `show_col_types = FALSE` to quiet this message.
└ @ RCall ~/.julia/packages/RCall/fTLHT/src/io.jl:166


RObject{VecSxp}
# A tibble: 1,744,977 x 8
      ID        Y     X1      X2     X3      X4      Z1      Z2
   <dbl>    <dbl>  <dbl>   <dbl>  <dbl>   <dbl>   <dbl>   <dbl>
 1     1   9.52    0.202 -0.463   0.798  0.734   0.685  -0.570 
 2     1  24.4     1.59  -1.95    1.20   1.43    1.64    0.369 
 3     1  -1.99    0.378 -0.0367  1.63  -1.15   -0.818   2.83  
 4     1 -17.4    -1.88   0.375  -0.498 -0.253   1.56    1.68  
 5     1  -0.0704  0.658 -0.165   0.780 -1.23   -0.0288 -1.09  
 6     1  -0.853   0.458 -0.313  -0.512 -0.800  -0.331   1.98  
 7     1  -1.80    0.220  0.328   1.32  -1.01   -0.363  -0.0703
 8     1   5.88    1.30   0.889  -0.854  0.0714 -0.658  -0.0339
 9     1  -9.21   -1.43  -0.522  -0.119 -0.580  -0.155  -1.89  
10     1 -11.3    -0.468 -0.700   0.872 -1.82    1.80    0.492 
# i 1,744,967 more rows


In [34]:
R"""
rtime <- system.time(mmod <- 
  lmer(Y ~ X1 + X2 + X3 + X4 + (1 + Z1 + Z2 | ID), testdata, REML = FALSE))
"""

RObject{RealSxp}
   user  system elapsed 
 74.009   7.134  81.715 


In [35]:
R"""
rtime <- rtime["elapsed"]
summary(mmod)
rlogl <- logLik(mmod)
"""
runtime[2] = @rget rtime
loglike[2] = @rget rlogl;

### MixedModels.jl


In [36]:
testdata = CSV.File("lmm_data.csv", types = Dict(1=>String)) |> DataFrame

Row,ID,Y,X1,X2,X3,X4,Z1,Z2
,String,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,1,9.52102,0.201821,-0.463234,0.797731,0.73357,0.685476,-0.569622
2,1,24.4063,1.58557,-1.94608,1.19787,1.43149,1.63962,0.369053
3,1,-1.99215,0.378332,-0.0367002,1.63072,-1.15031,-0.817843,2.83422
4,1,-17.4233,-1.8826,0.374561,-0.49786,-0.253248,1.56433,1.67857
5,1,-0.0704245,0.658283,-0.165487,0.77951,-1.22763,-0.0287779,-1.09172
6,1,-0.853357,0.457784,-0.313387,-0.512299,-0.800278,-0.330632,1.97609
7,1,-1.80061,0.220461,0.327879,1.32209,-1.01336,-0.362947,-0.0703055
8,1,5.88119,1.30135,0.88884,-0.853941,0.0714372,-0.658202,-0.0338648
9,1,-9.20504,-1.43248,-0.521638,-0.119287,-0.579596,-0.154869,-1.88707


In [37]:
mj = fit(MixedModel, @formula(Y ~ X1 + X2 + X3 + X4 + (1 + Z1 + Z2 | ID)), testdata)
bm_mm = @benchmark fit(MixedModel, @formula(Y ~ X1 + X2 + X3 + X4 + (1 + Z1 + Z2 | ID)), $testdata)
loglike[3] = loglikelihood(mj)
runtime[3] = median(bm_mm).time / 1e9

┌ Warning: ProgressMeter by default refresh meters with additional information in IJulia via `IJulia.clear_output`, which clears all outputs in the cell. 
│  - To prevent this behaviour, do `ProgressMeter.ijulia_behavior(:append)`. 
│  - To disable this warning message, do `ProgressMeter.ijulia_behavior(:clear)`.
└ @ ProgressMeter ~/.julia/packages/ProgressMeter/N660J/src/ProgressMeter.jl:607
Minimizing 73    Time: 0:00:00 ( 3.46 ms/it)


0.693040292

In [38]:
display(bm_mm)
mj

BenchmarkTools.Trial: 8 samples with 1 evaluation per sample.
 Range (min … max):  422.021 ms … 967.613 ms  ┊ GC (min … max):  1.83% … 58.83%
 Time  (median):     693.040 ms               ┊ GC (median):    43.02%
 Time  (mean ± σ):   672.622 ms ± 216.894 ms  ┊ GC (mean ± σ):  39.22% ± 22.75%

  ▁  ▁ ▁                      ▁   █                           █  
  █▁▁█▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  422 ms           Histogram: frequency by time          968 ms <

 Memory estimate: 886.92 MiB, allocs estimate: 2783.

|             |    Est. |     SE |        z |      p |   σ_ID |
|:----------- | -------:| ------:| --------:| ------:| ------:|
| (Intercept) |  0.0777 | 0.0434 |     1.79 | 0.0738 | 1.3744 |
| X1          |  6.5000 | 0.0009 |  6998.36 | <1e-99 |        |
| X2          | -3.4991 | 0.0009 | -3766.20 | <1e-99 |        |
| X3          |  1.0006 | 0.0009 |  1078.77 | <1e-99 |        |
| X4          |  5.0003 | 0.0009 |  5391.00 | <1e-99 |        |
| Z1          |         |        |          |        | 1.0482 |
| Z2          |         |        |          |        | 1.0185 |
| Residual    |  1.2246 |        |          |        |        |


### Summary

In [ ]:
using PrettyTables
using Printf

table_out = hcat(
    string.(method),
    [@sprintf("%.2f", x) for x in runtime],
    [@sprintf("%.6f", x) for x in loglike]
)

pretty_table(
    table_out;
    column_labels = ["Method", "Runtime", "Log-Like"]
)

┌────────────────┬─────────┬─────────────────┐
│         Method │ Runtime │        Log-Like │
├────────────────┼─────────┼─────────────────┤
│      My method │    0.05 │ -2840594.449982 │
│           lme4 │   82.37 │ -2840594.449980 │
│ MixedModels.jl │    0.71 │ -2840594.449982 │
└────────────────┴─────────┴─────────────────┘


## Q10. Be proud of yourself

Go to your resume/cv and claim you have experience performing analysis on complex longitudinal data sets with millions of records. And you beat current software by XXX fold.